--------------------------------------------------
# **Create Czech text dataset for training**
-------------------------------------------------

In [1]:
!apt install git-lfs




git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 138 not upgraded.


## ***Imports***

In [2]:
import os, sys
import zipfile
import subprocess
import io
import shutil
import json
import polars as pl
import numpy as np
import pyarrow as pa
import pickle

import pyarrow.parquet as pq
import pyarrow.dataset as ds

from huggingface_hub import (
    get_full_repo_name,
    login,
    upload_folder,
    hf_hub_download,
    HfApi
)


from datasets import load_dataset
from kaggle_secrets import UserSecretsClient
from tqdm import tqdm

## ***Hugging Face Login***

In [3]:
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
user_email = user_secrets.get_secret("user_email")
user_name = user_secrets.get_secret("user_name")

login(token=hf_token)

### ***Data files***

1. ***CZLC/cermat_czech_mc***
   
   The **Cermat Czech MultiChoice** dataset was collected from assignments official CERMAT website. The dataset was collected from three tiers of assignments: 6 year, 9 year primary school test and final high school tests (so-called maturita). The assignments were semi-manually extracted from official PDFs available at [CERMAT's website](https://maturita.cermat.cz/menu/testy-a-zadani-z-predchozich-obdobi/matematika/testy-a-zadani-matematika).

    **Collection Date Range:** years 2016-2023

    ***Licensing and Credits***

    The majority of collection work was done by our student co-worker Jan Kapsa. Members of CZLC do not own the test assignments, neither are responsible for their contents.

    **Source:** https://huggingface.co/datasets/CZLC/cermat_czech_mc
 
2. ***hynky/czech-justice-summ-alpaca-long***

    **Source:** https://huggingface.co/datasets/hynky/czech-justice-summ-alpaca-long

3. ***hynky/czech_news_dataset_v2***

    Dataset containing the news articles from major online news outlets collected from 2000-2022.
    Follow-up paper https://arxiv.org/abs/2307.10666 (v1 of the dataset)

    Changes from v1:
        - Better contribution of novinky.cz in later stages
        - More articles, as a mistake in filtering was fixed.

   Collection was done using CmonCrawl.

   The dataset should be used for Research only purposes as I don't have rights for articles itself.

   **Source:** https://huggingface.co/datasets/hynky/czech_news_dataset_v2

4. ***davidadamczyk/czechbench_czech_news***

   This is a simplified and subsampled test subset from the original [czech_news_dataset_v2](https://huggingface.co/datasets/hynky/czech_news_dataset_v2).

   **Source:** https://huggingface.co/datasets/davidadamczyk/czechbench_czech_news

5. ***CIIRC-NLP/czech_news_simple-cs***

   **Source:** https://huggingface.co/datasets/CIIRC-NLP/czech_news_simple-cs

   This is a simplified and subsampled test subset from the original czech_news_dataset_v2.

   Only 5 basic news categories are considered:

   - Foreign
   - Local
   - Sports
   - Economy

   The test set includes 200 examples per category, 1000 examples in total. Apart from the category label, each example also contains the article's headline, brief summary, full textual content, optional keywords, original category specification, and URL.

   This dataset was created for use within the [Czech-Bench](https://gitlab.com/jirkoada/czech-bench) evaluation framework.


In [4]:
NUM_GENERATED_SAMPLES = 250000

In [5]:
ds01 = load_dataset("CZLC/cermat_czech_mc")
ds02 = load_dataset("hynky/czech-justice-summ-alpaca-long")
ds03 = load_dataset("hynky/czech_news_dataset_v2")
ds04 = load_dataset("davidadamczyk/czechbench_czech_news")
ds05 = load_dataset("CIIRC-NLP/czech_news_simple-cs", "default")


README.md: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/649 [00:00<?, ? examples/s]

README.md:   0%|          | 0.00/503 [00:00<?, ?B/s]

data/train-00000-of-00001-93f855e5cee8b5(…):   0%|          | 0.00/12.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/20 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/980 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001-8fd5c2a953da2a2(…):   0%|          | 0.00/2.67M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
print("ds01:",type(ds01),'\n',ds01,'\n')
print("ds02:",type(ds02),'\n',ds02,'\n')
print("ds03:",type(ds03),'\n',ds03,'\n')
print("ds04:",type(ds04),'\n',ds04,'\n')
print("ds05:",type(ds05),'\n',ds05,'\n')

ds01: <class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    test: Dataset({
        features: ['type', 'query', 'choices', 'gold', 'context', 'subject', 'difficulty', 'source'],
        num_rows: 649
    })
}) 

ds02: <class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['output', 'instruction'],
        num_rows: 4560
    })
}) 

ds03: <class 'datasets.dataset_dict.DatasetDict'> 
 DatasetDict({
    train: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 1641471
    })
    validation: Dataset({
        features: ['url', 'authors', 'headline', 'brief', 'keywords', 'category', 'content', 'comments_num', 'server', 'category_unclean', 'authors_gender', 'authors_cum_gender', 'day_of_week', 'date'],
        num_rows: 144836
    })
    test: Dataset({
       

### ***We display data files in polars***

In [7]:
df01 = pl.from_arrow(ds01["test"].data.table)
df01.sample(5)

type,query,choices,gold,context,subject,difficulty,source
str,str,list[str],i64,str,str,str,str
"""MC""","""Ve kterém z následujících úsek…","[""A. mám horečku zlatou"", ""B. že stloukám si kříž"", … ""D. kraj pod sněhem mlčí""]",3,"""<bold>TEXT 1 (1)</bold> Jdu s …","""český jazyk a literatura""","""čtyřleté obory a nástavbová st…","""1. řádný termín 2018"""
"""MC""","""Ve kterém z následujících úsek…","[""A. její aktivity se zaměřují i na oblast"", ""B. říká matka jedné žákyně"", … ""D. být striktně rozděleny""]",0,"""<bold>Konec chlapců, dívek a s…","""český jazyk a literatura""","""maturitní zkouška""","""podzim 2022"""
"""MC""","""Které z následujících tvrzení …","[""A. V první části výchozího textu je podtržena věta jednoduchá, v druhé části výchozího textu je podtrženo souvětí."", ""B. Ve výchozím textu jsou podtrženy dvě věty jednoduché."", … ""D. V první části výchozího textu je podtrženo souvětí, v druhé části výchozího textu je podtržena věta jednoduchá.""]",1,"""<bold>(1)</bold> <underline>Pr…","""český jazyk a literatura""","""maturitní zkouška""","""podzim 2019"""
"""MC""","""Ve kterém z následujících úsek…","[""A. starobylá tradice má ovšem i své četné odpůrce"", ""B. ani místní věřící tento rituál nepodporovali"", … ""D. muži převlečení za ďábly je přeskakují""]",2,"""(1) Ve španělské vesničce Cast…","""český jazyk a literatura""","""čtyřleté obory a nástavbová st…","""1. řádný termín 2022"""
"""MC""","""Které z následujících tvrzení …","[""A. Ve stejné době jako on tvořil ruský spisovatel Anton Pavlovič Čechov."", ""B. Je mimo jiné autorem pohádek, v nichž některé z postav zemřou."", … ""D. Věnoval se mimo jiné dramatické tvorbě, psal např. konverzační komedie.""]",2,"""<bold>TEXT 1 Les Ballets Buben…","""český jazyk a literatura""","""maturitní zkouška""","""jaro 2018"""


In [8]:
df02 = pl.from_arrow(ds03["train"].data.table)
df02.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.novinky.cz/zahrani…",[],"""Francouzský satirický magazín …","""Francouzský satirický magazín …","[""Karikatura"", ""Mohamed"", … ""Evropa""]",1,"""Na obálce časopisu je ortodoxn…",null,4,"""Zahraniční""",[],0,3,2012-09-19 00:00:00
"""https://www.novinky.cz/zahrani…","[""Ivan Vilček""]","""Bush má promluvit ke Slovákům …","""Americký prezident George W. B…","[""Slovensko"", ""Bratislava"", … ""Zahraniční""]",1,"""""Oficiální program návštěvy pr…",null,4,"""Zahraniční""",[1],1,5,2005-02-11 00:00:00
"""https://berounsky.denik.cz/zpr…","[""Jakub Šťástka""]","""KRÁTCE: V Hořovicích se rozlou…","""Loučení se prázdninami je přic…","[""Hořovice"", ""Keř"", … ""Šelmberk (okres tachov)""]",0,"""Keře bobkovišní si budou moci …",null,5,"""ZPRÁVY""",[1],1,3,2019-08-28 00:00:00
"""https://zdarsky.denik.cz/zprav…","[""Helena Zelená Křížová""]","""Mimo tři stanovená místa a čas…","""Nové Město na Moravě – Novoměs…","[""Horácké muzeum"", ""Zubří"", … ""Michal šmarda""]",0,"""„Omezili jsme počet míst, kter…",null,5,"""Žďársko""",[2],2,4,2019-08-15 00:00:00
"""https://www.irozhlas.cz/zpravy…","[""Milan Kopp"", ""Ondřej Houska""]","""Za cestu do Spojených států mo…","""Evropané by měli za cestu do S…","[""Usa"", ""Víza"", ""Amerika""]",1,"""Přesně tolik by museli zaplati…",null,6,"""Zprávy ze světa""","[1, 1]",1,4,2009-09-10 00:00:00


In [9]:
df03_train = pl.from_arrow(ds03["train"].data.table)
df03_train.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.novinky.cz/zahrani…",[],"""Dělohy dvou matek transplantov…","""Ve Švédsku byly u dvou mladých…","[""Děloha"", ""Transplantace"", … ""Evropa""]",1,"""""Ženám, které dostaly dělohu, …",null,4,"""Zahraniční""",[],0,2,2012-09-18 00:00:00
"""https://www.novinky.cz/domaci/…",[],"""Soudy s podmíněným propuštěním…","""Krajský soud v Plzni minulý tý…",[],2,"""Ten totiž stanovuje, že člověk…",null,4,"""Domácí""",[],0,3,2020-01-15 00:00:00
"""https://sokolovsky.denik.cz/z-…","[""Vladimír Meluzín""]","""Hlasování o „stavbu“ ukazuje, …","""Karlovarský kraj – Do finále s…","[""Hlasování"", ""Rusalka"", … ""Boží dar""]",0,"""Ze 16 přihlášených staveb zatí…",null,5,"""ZPRÁVY""",[1],1,3,2017-06-07 00:00:00
"""https://mostecky.denik.cz/hoke…","[""Václav Veverka""]","""„Snad jednou porodím v den, kd…","""Litvínov /ROZHOVOR/ – Eliška B…",[],3,"""Bílková je manželovi, kterého …",null,5,"""SPORT""",[1],1,3,2018-10-03 00:00:00
"""https://www.irozhlas.cz/region…","[""Simona Bartošová"", ""Josef Kopecký""]","""Tři 'Gentlemani silnic' z Chru…","""Chrudimsko má tři nové Gentlem…","[""Policie"", ""Nehoda"", … ""Česká pojišťovna""]",2,"""Nehoda se stala na začátku bře…",null,6,"""Regiony""","[2, 1]",3,4,2013-05-16 00:00:00


In [10]:
df03_validation = pl.from_arrow(ds03["validation"].data.table)
df03_validation.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://klatovsky.denik.cz/zpr…","[""Daniela Loudová""]","""Nový poldr zabrání zatopení za…","""Obyvatele Lub, kterým každý vě…","[""Luby"", ""Klatovy"", … ""Česko""]",0,"""„Obě zařízení jsou v režimu su…",null,5,"""ZPRÁVY""",[2],2,5,2020-06-26 00:00:00
"""https://www.denik.cz/nehody/tr…","[""Milan Holakovský""]","""Tragédie: Zaparkovaný kamion s…","""/FOTOGALERIE/ Život řidiče kam…","[""Okres praha-západ"", ""Chrášťany"", … ""Rudná""]",0,"""Zaparkovaný tahač s připojeným…",null,5,null,[1],1,3,2021-01-13 00:00:00
"""https://www.seznamzpravy.cz/cl…","[""Jiří Hošek""]","""Místo medailí zákopy a kulka. …","""Ve 30. letech jim u nohou leže…",[],1,"""Carl Ludwig „Luz” Long, Německ…",null,1,"""Svět""",[1],1,7,2020-05-10 00:00:00
"""https://sport.aktualne.cz/fotb…",[],"""Dočasná revoluce ve fotbale. T…","""Fotbalové týmy budou moci vzhl…","[""Fifa"", ""Soutěž"", … ""Revoluce""]",3,"""FIFA uvedla, že bude na jednot…",null,3,"""Fotbal""",[],0,5,2020-05-08 00:00:00
"""https://www.idnes.cz/fotbal/do…","[""Jiří Punčochář""]","""Bundesliga mezi kukuřicí. Fano…","""Původně to vypadlo jako vtip. …","[""Jihomoravský kraj"", ""Fc bayern mnichov"", … ""Křepice""]",3,"""„Telefonát přišel z německého …",32,2,"""Fotbal""",[1],1,3,2020-09-16 00:00:00


In [11]:
df03_test = pl.from_arrow(ds03["test"].data.table)
df03_test.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date
str,list[str],str,str,list[str],i64,str,i32,i64,str,list[i64],i64,i64,datetime[μs]
"""https://www.idnes.cz/plzen/zpr…","[""Jitka Šrámková"", ""Valentýna Bílá""]","""Na Tachovsku začne testování v…","""Hromadné testování ve firmách,…","[""Plzeňský kraj"", ""Ilona mauritzová"", … ""Trutnov""]",2,"""Tachovsko se tak mělo připojit…",6,2,"""Kraje""","[2, 2]",2,4,2021-02-18 00:00:00
"""https://www.idnes.cz/revue/spo…",[],"""Dívka ze slavného memu prodala…","""Zoe Rothová (21) prodala svou …","[""Aukce"", ""Celebrity"", … ""Digitální fotografie""]",5,"""Z malé holčičky je dnes podnik…",33,2,"""Revue""",[],0,5,2021-05-14 00:00:00
"""https://www.idnes.cz/zpravy/do…","[""Tomáš Matoušek""]","""Žák je v karanténě, ale rodiče…","""Kantoři učí podle svého běžnéh…","[""Miloš zeman"", ""Covid-19"", … ""Distanční výuka""]",2,"""Své zkušenosti s rodiči, kteří…",484,2,"""Domácí""",[1],1,3,2021-12-01 00:00:00
"""https://zpravy.aktualne.cz/dom…",[],"""Zavazadla vyhoštěných Rusů zap…","""V pondělí odpoledne odlétlo os…","[""Zavazadlo"", ""Rodina"", … ""Andrej babiš""]",2,"""Na záběrech jsou vidět krabice…",null,3,"""Domácí""",[],0,2,2021-04-20 00:00:00
"""https://www.seznamzpravy.cz/cl…",[],"""Kroměříž zahájila ostrý provoz…","""Kroměřížská radnice zahájila o…","[""Kroměříž"", ""Parkovací dům"", ""Turistická centra""]",2,"""Město uvedlo parkovací dům do …",null,1,"""Regiony""",[],0,4,2022-06-02 00:00:00


In [12]:
df04_train = pl.from_arrow(ds04["train"].data.table)
df04_train.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.irozhlas.cz/zpravy…",[],"""Ministerstvo poslalo ředitelům…","""Ministerstvo školství poslalo …","[""Koronavirus"", ""Koronavirus česko"", … ""Učitelé""]",2,"""Výběr pracovníků s přednostním…",null,6,"""Zprávy z domova""",[],0,3,2021-02-24 00:00:00,2,180
"""https://sumpersky.denik.cz/hok…","[""Miroslav Pergl""]","""Nájezdový specialista a perspe…","""Kádr Draků pro nadcházející pr…",[],3,"""„Pro Dávida Kohúta je důležité…",null,5,"""Hokej""",[1],1,7,2022-05-15 00:00:00,3,140
"""https://www.denik.cz/ekonomika…","[""Jiří Janda""]","""PŘEHLEDNĚ: 500 korun k důchodu…","""Stát od příštího roku přilepší…","[""Důchod"", ""Ekonomika"", … ""Vyčíslení""]",5,"""Na jakou částku se od ledna 20…",null,5,"""Ekonomika""",[1],1,4,2022-06-23 00:00:00,7,10
"""https://www.seznamzpravy.cz/cl…","[""Martin Čaban""]","""Vizita: Jak pošťouchnout k očk…","""Britský vědecký časopis Nature…","[""Newsletter vizita"", ""Zdravotnictví"", … ""Covid-19""]",2,"""Protože na stránkách asi nejpr…",null,1,"""Domácí""",[1],1,3,2022-06-08 00:00:00,2,131
"""https://decinsky.denik.cz/osta…","[""Jaroslav Zeman""]","""Sportovci z Děčínska se zlobí!…","""Pandemie koronaviru v Česku us…","[""Sport"", ""Okres děčín"", … ""Hc děčín""]",3,"""Ondřej Michálek, hlavní trenér…",null,5,"""SPORT""",[1],1,5,2021-05-07 00:00:00,3,32


In [13]:
df04_test = pl.from_arrow(ds04["test"].data.table)
df04_test.sample(5)

url,authors,headline,brief,keywords,category,content,comments_num,server,category_unclean,authors_gender,authors_cum_gender,day_of_week,date,__index_level_0__,__index_level_1__
str,list[str],str,str,list[str],i64,str,f64,i64,str,list[i64],i64,i64,datetime[μs],i64,i64
"""https://www.idnes.cz/brno/zpra…",[],"""Na D2 se u Blučiny srazily kam…","""Nehoda dvou nákladních vozů dn…","[""Jihomoravský kraj"", ""Dálnice d2"", … ""Nehoda""]",2,"""„Řidiči jezdí odklonem, sjíždě…",8.0,2,"""Kraje""",[],0,4,2022-06-23 00:00:00,2,115
"""https://www.idnes.cz/fotbal/do…",[],"""Fotbalisté Jablonce přešli v d…","""Fotbalisté Jablonce vyhráli v …","[""Fk jablonec"", ""Václav pilař"", … ""Pohár české pošty""]",3,"""„Vstoupili jsme do zápasu dobř…",5.0,2,"""Fotbal""",[],0,6,2021-03-27 00:00:00,3,17
"""https://zpravy.aktualne.cz/eko…",[],"""""Vyléčíme cokoliv."" Řada e-sho…","""Uplynulý rok s koronavirem ovl…","[""Covid-19"", ""Státní potravinářská inspekce"", … ""Mendelova univerzita v brně""]",5,"""Ředitel SZPI Martin Klanica ve…",null,3,"""Ekonomika""",[],0,3,2021-03-03 00:00:00,7,97
"""https://zpravy.aktualne.cz/zah…","[""Dominika Perlínová""]","""Seychely naočkovaly většinu li…","""Seychely naočkovaly největší č…","[""Vakcína"", ""Seychely"", … ""Indický oceán""]",1,"""Případy v posledních dnech zap…",null,3,"""Zahraničí""",[2],2,3,2021-05-12 00:00:00,1,148
"""https://zpravy.aktualne.cz/zah…","[""Jana Václavíková""]","""Když mu došly peníze, žebral. …","""V srpnu 1997 si dvouapůlletý K…","[""Čína"", ""Rodina"", … ""Jeden svět""]",1,"""Když malý Kuo zmizel, dával si…",null,3,"""Zahraničí""",[2],2,2,2021-07-13 00:00:00,1,23


In [14]:
df05 = pl.from_arrow(ds05["test"].data.table)
df05.sample(5)

url,headline,brief,keywords,category,content,category_unclean
str,str,str,list[str],i64,str,str
"""https://www.seznamzpravy.cz/cl…","""Lanovky v Krkonoších mimo prov…","""Lanovky ve střediscích Krkonoš…","[""Krkonoše"", ""Lanovka"", ""Turistika""]",2,"""Opět se lanovky v Krkonoších n…","""Regiony"""
"""https://www.seznamzpravy.cz/cl…","""Glosa: Další nudící se miliard…","""Rozmohl se nám tu takový nešva…","[""Prezidentské volby"", ""Tomáš březina""]",2,"""Po Karlu Janečkovi oznámil ve …","""Volby"""
"""https://magazin.aktualne.cz/ku…","""Velvyslanec koupil zámek i se …","""Herec a zpěvák Ondřej Havelka …","[""Klasická hudba"", ""Kultura"", … ""Divadlo antonína dvořáka""]",4,"""Šestašedesátiletý Havelka s os…","""Klasická hudba"""
"""https://www.novinky.cz/kultura…","""Ozvěny Ji.hlavy nabídnou čtyři…","""Čtyři desítky filmů z loňského…",[],4,"""„Filmový svět se kvůli pandemi…","""Kultura"""
"""https://www.idnes.cz/kultura/l…","""RECENZE: Čtenář má chuť poděko…","""Během roku nachodil po nejvyšš…","[""Ivan fíla"", ""Jeseníky"", … ""Jeseník""]",4,"""Filmař, spisovatel a fotograf …","""Kultura"""


### ***Creating a summary data file***

In [15]:
def build_unified_contexts(ds01, ds02, ds03, ds04, ds05):
    print("I am starting the unification of Czech text contexts using Polars...")
    
    # --- 1. News Corpus (Hynky v2 - 1.6M+ lines) ---
    # We glue the title, annotation and the entire body of the article into one massive block
    df_news_A = pl.from_arrow(ds03["train"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])
    df_news_B = pl.from_arrow(ds03["validation"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])
    df_news_C = pl.from_arrow(ds03["test"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])

    # --- 2. davidadamczyk/czechbench_czech_news ---
    df_news_D = pl.from_arrow(ds04["train"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])
    df_news_E = pl.from_arrow(ds04["test"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])

    # --- 3. CIIRC-NLP/czech_news_simple-cs ---
    df_news_F = pl.from_arrow(ds05["test"].data.table).lazy().select([
        (pl.col("headline") + "\n" + pl.col("brief") + "\n" + pl.col("content")).alias("context"),
        pl.lit("news").alias("source")
    ])
    
    # --- 4. Legal / Alpaca dataset (czech-justice-summ-alpaca-long) ---
    # Here it makes sense to combine the instruction and the answer into one continuous text
    df_alpaca = pl.from_arrow(ds02["train"].data.table).lazy().select([
        (pl.col("instruction") + "\n" + pl.col("output")).alias("context"),
        pl.lit("alpaca_justice").alias("source")
    ])
    
    # --- 5. CERMAT Dataset (cermat_czech_mc) ---
    # Here we have the 'context' column, which we can enrich with a question
    df_cermat = pl.from_arrow(ds01["test"].data.table).lazy().select([
        (pl.col("context") + "\nOtázka: " + pl.col("query")).alias("context"),
        pl.lit("cermat").alias("source")
    ])

    # --- 6. Concatenation of all streams ---
    # Polars performs vertical joins extremely fast without unnecessary duplicate memory allocation
    unified_lazy = pl.concat([df_news_A, df_news_B, df_news_C, df_news_D, df_news_E, df_news_F, df_alpaca, df_cermat])
    
    # --- 7. Text extraction and cleaning + Calculation of logical predicates for JLNN ---
    final_df = unified_lazy.filter(
        pl.col("context").is_not_null() & (pl.col("context").str.len_chars() > 50)
    ).select([
        # Clean resulting text (context)
        pl.col("context"),
        pl.col("source"),
        
        # [PREDICT A]: Detection of masculine nouns in the text (Axiom of subject-predicate agreement)
        pl.col("context").str.to_lowercase().str.contains_any(
            ["chlapci", "muži", "dělníci", "studenti", "hráči", "učitelé", "lidi", "psi"]
        ).cast(pl.Float32).alias("has_masc_animate"),
        
        # [PREDICATE C]: Detection of subordinating conjunctions introducing subordinate clauses (Axiom of Punctuation)
        pl.col("context").str.to_lowercase().str.contains_any(
            ["že", "protože", "aby", "který", "jelikož", "jakmile", "zatímco"]
        ).cast(pl.Float32).alias("has_subord_conjunction"),
        
        # Auxiliary check: Is there a comma present in the text? (Basis for evaluating the t-norm)
        pl.col("context").str.contains(",").cast(pl.Float32).alias("has_comma")
    ]).collect()
    
    print(f"Unification complete! Total number of entire contexts in dataset: {final_df.height:,}")
    return final_df

In [16]:
%%time
df_dataset = build_unified_contexts(ds01, ds02, ds03, ds04, ds05)
df_dataset.sample(5)

I am starting the unification of Czech text contexts using Polars...
Unification complete! Total number of entire contexts in dataset: 1,938,272
CPU times: user 1min 37s, sys: 23.6 s, total: 2min
Wall time: 1min 48s


context,source,has_masc_animate,has_subord_conjunction,has_comma
str,str,f32,f32,f32
"""Majovák zahrál v Karviné Novor…","""news""",0.0,1.0,1.0
"""RECENZE: V přetěžké úloze Kňaž…","""news""",0.0,1.0,1.0
"""Kotík nabídl výjimečný zážitek…","""news""",0.0,1.0,1.0
"""Petanque v zimě? Proč by ne...…","""news""",0.0,1.0,1.0
"""Na Jižní spojce se srazilo aut…","""news""",0.0,1.0,1.0


In [17]:
df_dataset.shape

(1938272, 5)

### ***Preparing the RAW dataset (Pure sentence extraction in Polars)***

In [18]:
def create_raw_sentence_dataset(unified_df):
    print("I extract clean sentence segments and classify their punctuation...")
    
    # This regex finds any characters that end with either a period, question mark, exclamation mark, comma,
    # or at the end of the entire text. This will keep the sign right at the end of the segment!
    regex_segment_with_punc = r"[^.\?!,]+[\.\?!,]?"

    raw_segments = (
        unified_df
        # 1. We extract all the matching sentence segments as a Word List
        .select(
            pl.col("context").str.extract_all(regex_segment_with_punc)
        )
        # 2. Expand the sheet into separate rows
        .explode("context")
        .rename({"context": "segment"})
        
        # 3. We clean up spaces at the beginning and end (the stuck punctuation will remain)
        .with_columns(pl.col("segment").str.strip_chars())
        
        # 4. FILTERING: We remove empty or too short things
        .filter(
            (pl.col("segment").str.len_chars() > 5) & 
            (pl.col("segment").str.len_chars() < 200) &
            pl.col("segment").str.contains(r"[a-zA-Zá-žÁ-Ž]")
        )
        
        # 5. CLASSIFICATION OF TERMINAL TYPE
        .with_columns([
            pl.when(pl.col("segment").str.ends_with("?")).then(pl.lit("otazník"))
            .when(pl.col("segment").str.ends_with("!")).then(pl.lit("vykřičník"))
            .when(pl.col("segment").str.ends_with(".")).then(pl.lit("tečka"))
            .when(pl.col("segment").str.ends_with(",")).then(pl.lit("čárka"))
            .otherwise(pl.lit("žádná"))
            .alias("punctuation_type")
        ])
        
        # 6. Random sample
        .sample(n=NUM_GENERATED_SAMPLES, with_replacement=False, seed=42)
    )
    
    print(f"Prepared {raw_segments.height} sentence segments.")
    return raw_segments

In [19]:
%%time
raw_sentence_df = create_raw_sentence_dataset(df_dataset)
raw_sentence_df.head(5)

I extract clean sentence segments and classify their punctuation...
Prepared 250000 sentence segments.
CPU times: user 2min 19s, sys: 13.3 s, total: 2min 33s
Wall time: 2min 9s


segment,punctuation_type
str,str
"""Učinily zásadní krok k záchran…","""tečka"""
"""Úřad práce,""","""čárka"""
"""že podle vypracovaného modelu …","""tečka"""
"""Podporu rozpočtu naopak vyjádř…","""tečka"""
"""Foto: Miroslav Homola,""","""čárka"""


In [20]:
raw_sentence_df.shape

(250000, 2)

### ***Export dataset***

In [21]:
def export_polars_to_pyarrow_dataset(polars_df, output_dir="processed_dataset"):
    print("Converting Polars DataFrame to PyArrow Table...")
    
    # We define a dictionary for translating Czech values into English folder names
    translation_dict = {
        "otazník": "question_mark",
        "vykřičník": "exclamation_mark",
        "tečka": "period",
        "čárka": "comma",
        "žádná": "none"
    }
    
    # Temporarily add a column for folders in Polars and immediately convert to Arrow
    arrow_table = (
        polars_df
        .with_columns(
            pl.col("punctuation_type").replace(translation_dict).alias("folder_name")
        )
        .to_arrow()
    )
    
    print(f"PyArrow table ready. Number of rows: {arrow_table.num_rows:,}")
    os.makedirs(output_dir, exist_ok=True)
    
    print(f"Saving PyArrow Dataset to folder '{output_dir}' with English names...")
    
    # Writing to disk - partition will be done according to "folder_name" (English)
    ds.write_dataset(
        data=arrow_table,
        base_dir=output_dir,
        format="parquet",
        partitioning=["folder_name"],
        use_threads=True,
        existing_data_behavior="overwrite_or_ignore"
    )
    
    print("Export completed successfully!")

In [22]:
%%time
export_polars_to_pyarrow_dataset(raw_sentence_df, output_dir="raw_segments_dataset")

Converting Polars DataFrame to PyArrow Table...
PyArrow table ready. Number of rows: 250,000
Saving PyArrow Dataset to folder 'raw_segments_dataset' with English names...
Export completed successfully!
CPU times: user 287 ms, sys: 68.9 ms, total: 356 ms
Wall time: 192 ms
